# Multimodal Hateful Meme Detection — Colab Training

**Author:** Jagadeesh Venkatakumar · CMPE 257

This notebook trains and evaluates **five** approaches on the [Facebook Hateful Memes](https://www.kaggle.com/datasets/parthplc/facebook-hateful-meme-dataset) dataset:

| # | Model | Training |
|---|--------|----------|
| 1 | Zero-shot CLIP | None (baseline) |
| 2 | Frozen CLIP + MLP | Train MLP only (CLIP frozen) |
| 3 | CLIP + BERT fusion | Train BERT + fusion |
| 4 | CLIP + BERT + cross-attention | Train BERT + attention |
| 5 | CLIP + MLP + **top-layer CLIP fine-tune** | Train MLP + last 2 ViT blocks (ViT-L/14) |

---

## Before you start

1. **Runtime → Change runtime type → GPU** (T4 or better)
2. Get a **Kaggle API token**: [kaggle.com/settings](https://www.kaggle.com/settings) → API → Create New Token
3. Accept the dataset terms on Kaggle (open the dataset page once while logged in)
4. **Do not** commit your real Kaggle token to GitHub — paste it only in Colab

## Step 1 — Clone project from GitHub

Downloads all Python code (`src/train.py`, models, etc.).  
If you already cloned this session, skip clone and only run `%cd` line.

In [ ]:
# Clone repository (run once per Colab session)
import os
if not os.path.isdir('Hateful-Meme-Detection-Using-CLIP'):
    !git clone https://github.com/pjvk/Hateful-Meme-Detection-Using-CLIP.git
%cd Hateful-Meme-Detection-Using-CLIP
!git pull

## Step 2 — Install dependencies

Installs PyTorch, OpenAI CLIP, Hugging Face `transformers` (BERT), and other packages.

In [ ]:
!pip install -q -r requirements.txt

## Step 3 — Kaggle API token

Paste your token below (replace the placeholder).  
Alternative: Colab **Secrets** → name `KAGGLE_API_TOKEN` → use `userdata.get(...)`.

In [ ]:
import os

# Replace with your token from https://www.kaggle.com/settings
os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_API_TOKEN_HERE"

## Step 4 — Download dataset from Kaggle

Downloads ~3GB, links folder as `./data` with `img/`, `train.jsonl`, `dev.jsonl`, `test.jsonl`.  
**First run may take several minutes.**

In [ ]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("parthplc/facebook-hateful-meme-dataset")
print("Download path:", path)

# Find folder containing dev.jsonl + img/
root = next(Path(path).rglob("dev.jsonl")).parent

if Path("data").is_symlink():
    Path("data").unlink()
elif Path("data").is_dir():
    import shutil
    shutil.rmtree("data")
elif Path("data").exists():
    Path("data").unlink()

Path("data").symlink_to(root.resolve())
print("Linked data →", Path("data").resolve())
!ls data

## Step 5 — Verify dataset & GPU

In [ ]:
!head -n 1 data/dev.jsonl
!python -m src.verify_data --data-dir data --split dev

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Enable GPU in Runtime settings for practical training times.")

---

## Model 1 — Zero-shot CLIP (baseline)

Uses pretrained CLIP **without training**. Compares meme image+text embeddings to hand-written class prompts.  
Expect modest accuracy; useful as a lower bound in your report.

In [ ]:
!python -m src.zeroshot --data-dir data --split dev --clip-model ViT-B/32

---

## Model 2 — Frozen CLIP + MLP

- **CLIP ViT-B/32** encodes image and text (frozen)
- Embeddings are concatenated → **MLP classifier** (trained)
- Fast training; competitive accuracy with limited compute

Checkpoint: `checkpoints/clip_mlp.pt`

In [ ]:
!python -m src.train --model clip_mlp --data-dir data --epochs 10 --batch-size 32 --lr 1e-3 --clip-model ViT-B/32

In [ ]:
!python -m src.evaluate --model clip_mlp --data-dir data --split dev --checkpoint checkpoints/clip_mlp.pt

---

## Model 3 — CLIP + BERT fusion (no cross-attention)

- **CLIP** for images (frozen)
- **BERT** for meme text (trainable)
- Concatenate projected features → MLP

Uses more GPU memory; default batch size 16.  
Checkpoint: `checkpoints/clip_bert.pt`

In [ ]:
!python -m src.train --model clip_bert --data-dir data --epochs 10 --batch-size 16 --lr 5e-5 --clip-model ViT-B/32

In [ ]:
!python -m src.evaluate --model clip_bert --data-dir data --split dev --checkpoint checkpoints/clip_bert.pt --batch-size 16

---

## Model 4 — CLIP + BERT + cross-attention

- Image embedding **attends** to BERT token sequence
- Captures interactions between visual and language cues
- Heavier model; longest training time

Checkpoint: `checkpoints/clip_bert_cross.pt`

In [ ]:
!python -m src.train --model clip_bert_cross --data-dir data --epochs 12 --batch-size 16 --lr 2e-5 --clip-model ViT-B/32

In [ ]:
!python -m src.evaluate --model clip_bert_cross --data-dir data --split dev --checkpoint checkpoints/clip_bert_cross.pt --batch-size 16

---

## Model 5 — CLIP + MLP with top-layer CLIP fine-tuning (ViT-L/14)

Targets **higher dev accuracy** by:

- **ViT-L/14** backbone (default for this model)
- Fine-tuning **last 2 visual transformer blocks** only (CLIP LR `5e-6`, MLP LR `1e-4`)
- **`--class-weights`** for imbalanced labels
- Saves to `checkpoints/clip_mlp_ft.pt` (separate from Model 2)

Use batch 16; if CUDA OOM, add `--batch-size 8` to the train command.

In [ ]:
!python -m src.train --model clip_mlp_ft --data-dir data --class-weights

In [ ]:
!python -m src.evaluate --model clip_mlp_ft --data-dir data --split dev --checkpoint checkpoints/clip_mlp_ft.pt --batch-size 16

---

## Step 6 — Download checkpoints (optional)

Weights are **not** stored on GitHub. Download to your PC for local API / report.  
Uncomment the checkpoint(s) you need.

In [ ]:
from google.colab import files

files.download('checkpoints/clip_mlp.pt')
files.download('checkpoints/clip_mlp_ft.pt')
# files.download('checkpoints/clip_bert.pt')
# files.download('checkpoints/clip_bert_cross.pt')

---

## Report tips

Build a table from evaluate output:

| Model | Dev accuracy | F1 | Training time notes |
|-------|--------------|-----|---------------------|
| Zero-shot CLIP | | | No training |
| CLIP + MLP | | | Frozen CLIP, faster training |
| CLIP + BERT | | | Dual encoder |
| CLIP + BERT + cross-attn | | | |
| CLIP + MLP + top-layer fine-tune | | | ViT-L/14, targets higher accuracy |

Discuss **accuracy vs training cost** — not which model is forced to be "best."